# [P2] Q2 A + B: data audit and cleaning

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q2_laptop_analysis.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q2a` (audit), `t_q2b` (cleaning and new variables, with validation).

Decisions to read on the team board first (board ids): `q_data q_dups q_issues` (A), `os_map st_type rare validate` (B), and the shared `st_extra`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


# Q2 A: Data audit

Before analysing the laptop data we investigate its structure and quality. This notebook section applies the team decisions below.

| Decision | What we chose |
|---|---|
| `q_data` | The dataset is a copy in `data/laptop_price_dataset.csv` (no space in the name). The raw file is never modified. |
| `q_dups` | Near-duplicate rows are kept and documented; at the end we check that the conclusions hold without them. |
| `q_issues` | Which problems we address and which we only document (see the register in section 6). |
| `q_audit` | The audit contains: a variables table, a statement of what a row is, data type checks, range and unit checks, and consistency checks between columns. |
| `g_lang`, `g_nb` | English, and every step is explained in a text cell before its code. |

## 0. Setup

We import the libraries and read the file with `load_raw()`. This function only reads: no column is changed, no row is dropped. Keeping the raw table untouched lets us always compare against the original.

In [ ]:
import numpy as np
import pandas as pd

from q2_clean import load_raw

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

raw = load_raw()
print("shape (rows, columns):", raw.shape)
raw.head(3)

## 1. What does one row represent?

Decision (`q_audit`, row definition): we state in one sentence what a row is, because it determines what counts as one observation.

**Each row is a laptop listing: one configuration (brand, model, screen, CPU, RAM, storage, GPU, operating system, weight) together with one price in euros.** A row is not necessarily a unique model: the same configuration can appear several times with different prices. We check this in section 4.

## 2. Variables table

Decision (`q_audit`, variables table): one row per column with its pandas data type, its kind, the number of different values, the number of missing values, and an example. The *kind* is assigned by us from the meaning of the column, and `assert` makes sure that no column is left unclassified.

- **numeric**: values we can compute with (price, weight, RAM...).
- **categorical**: labels or types, with no arithmetic meaning (company, operating system...).
- **structured text**: text that contains several pieces of information in one string and has to be parsed before use (for example `Memory` holds both capacity and storage type).

In [ ]:
NUMERIC = ["Inches", "CPU_Frequency (GHz)", "RAM (GB)", "Weight (kg)", "Price (Euro)"]
CATEGORICAL = ["Company", "TypeName", "CPU_Company", "GPU_Company", "OpSys"]
STRUCTURED_TEXT = ["Product", "ScreenResolution", "CPU_Type", "Memory", "GPU_Type"]

kind = {c: "numeric" for c in NUMERIC}
kind.update({c: "categorical" for c in CATEGORICAL})
kind.update({c: "structured text" for c in STRUCTURED_TEXT})
assert set(kind) == set(raw.columns), "every column must be classified exactly once"

variables = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "kind": pd.Series(kind).reindex(raw.columns),
    "n_unique": raw.nunique(),
    "n_missing": raw.isna().sum(),
    "example": raw.iloc[0],
})
variables

**What the table shows**

- The table has 1,275 rows and 15 columns: 5 numeric, 5 categorical and 5 structured-text columns.
- **No column has missing values** (`n_missing` is 0 everywhere).
- The numeric columns already have numeric types (`float64` or `int64`), so they can be used in calculations as they are. The text columns have the pandas text type (`str`).
- Several text columns have many different values: `Product` (618), `GPU_Type` (106), `CPU_Type` (93), `ScreenResolution` (40) and `Memory` (39). They mix several pieces of information in one string, and `Memory` in particular has to be parsed to get the storage capacity and the storage type (section 2 of the cleaning notebook, task Q2 B).
- `Price (Euro)` has 791 different values among 1,275 rows: prices repeat, which is expected when the same configuration is listed more than once. We look at this in section 4.